# Download das bibliotecas

In [16]:
!pip install numpy==1.23.5 pandas==2.2.2

In [17]:
!pip install pmdarima

In [18]:
!pip install pyswarm

# Importacao dos dados

In [19]:
import pandas as pd
import os
import numpy as np
import random

In [20]:

from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [21]:
caminho_dados = '/content/drive/Shareddrives/IC - Otimizacao computacional para inflacao/Dados/dados_USA'

# Listar os arquivos dentro da pasta "Dados"
arquivos_dados = os.listdir(caminho_dados)

print(arquivos_dados)


['Untitled0.ipynb', 'dados_macroeconomicos_usa.csv']


In [22]:
# Caminho para o arquivo CSV
caminho_arquivo = os.path.join(caminho_dados, 'dados_macroeconomicos_usa.csv')

# Ler o arquivo CSV
df = pd.read_csv(caminho_arquivo)

In [23]:
df = df.set_index('DATE')
df.drop(columns=['Unnamed: 0'], inplace=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 180 entries, 2010-01-01 to 2024-12-01
Data columns (total 19 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Inflacao_CPI                 180 non-null    float64
 1   Inflacao_CPI_Core            180 non-null    float64
 2   PCE_Core                     180 non-null    float64
 3   Desemprego                   180 non-null    float64
 4   Desemprego_menos_27_semanas  180 non-null    float64
 5   Taxa_Juros_Fed               180 non-null    float64
 6   Emprego_Total_Nao_Agricola   180 non-null    int64  
 7   Permissoes_Construcao        180 non-null    int64  
 8   Salario_Medio                180 non-null    float64
 9   Salario_Medio_Real_Hora      180 non-null    float64
 10  Producao_Industrial          180 non-null    float64
 11  Capacidade_Instalada         180 non-null    float64
 12  PCE                          180 non-null    float64
 13  Deflator_

# Importacao das ferramentas para implementacao do PSO

In [24]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler
from pyswarm import pso  # Biblioteca para PSO

# Funcao fitness

In [25]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import numpy as np

# 📌 Função de fitness (não foi alterada)
def feature_selection_fitness(mask, X_train_scaled, y_train, X_test_scaled, y_test, model, metric_func):
    """
    Função de avaliação para PSO com modelo passado como argumento e métrica.
    """
    mask = np.round(mask).astype(int)  # Converter para 0 ou 1
    selected_features = X_train_scaled.columns[mask == 1]  # Selecionar as features

    if len(selected_features) == 0:  # Caso não tenha selecionado nenhuma feature
        return 9999  # Penalização por não selecionar nenhuma feature

    # Treinar o modelo com as features selecionadas
    model.fit(X_train_scaled[selected_features], y_train)

    # Fazer previsões
    y_pred = model.predict(X_test_scaled[selected_features])

    # Calcular a métrica
    score = metric_func(y_test, y_pred)

    return score  # Retorna a métrica calculada (MAE, MSE, etc.)

# Divisao dos dados e normalizacao dos dados

In [26]:

# 📌 Preparação dos dados
X = df.drop(columns=['Inflacao_CPI'])  # Removendo Inflacao_CPI das features
y = df['Inflacao_CPI']

# Divisão entre treino e teste (SÉRIE TEMPORAL, sem shuffle)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=False)

Normalizacao dos dados(media 0 e desvio padrao 1)

In [27]:
# 📌 Normalização dos dados
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)


# Funcoes auxiliares

In [28]:
import numpy as np
import pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX
from pmdarima import auto_arima
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import joblib

# Funções auxiliares

def make_stationary(df, max_diff=5):
    """Torna a série temporal estacionária através de diferenciação."""
    df_diff = df.copy()
    for i in range(1, max_diff + 1):
        stationary = True
        for column in df_diff.columns:
            if not check_stationarity(df_diff[[column]]):
                stationary = False
                df_diff[[column]] = df_diff[[column]].diff().fillna(method='ffill').fillna(method='bfill')
        if stationary:
            print(f"Série estacionária após {i-1} diferenciações.")
            return df_diff.dropna()
    raise ValueError("A série não se tornou estacionária após o número máximo de diferenciações.")

def check_stationarity(series, significance_level=0.05):
    """Verifica se a série é estacionária usando o teste ADF."""
    from statsmodels.tsa.stattools import adfuller
    result = adfuller(series.dropna())
    return result[1] < significance_level

def plot_forecasts(y_test, final_forecast, title='Previsão Dinâmica'):
    """Plota as previsões comparadas com os valores reais."""
    plt.figure(figsize=(15, 3))
    plt.plot(y_test.index, y_test, label='Valores Reais', color='blue')
    plt.plot(y_test.index, final_forecast, label='Previsão', color='orange')
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.show()


# Arimax

In [29]:
import warnings
warnings.filterwarnings('ignore')

In [30]:
def executar_modelo_arimax(df, exogenous_vars, tecnica_nome):
    # Tornar o DataFrame estacionário
    df_stationary = make_stationary(df[['Inflacao_CPI'] + exogenous_vars])

    # Dividir em treino e teste
    train_size = int(len(df_stationary) * 0.8)
    train, test = df_stationary.iloc[:train_size], df_stationary.iloc[train_size:]

    # Variável dependente e exógenas
    y_train = train['Inflacao_CPI']
    y_test = test['Inflacao_CPI']
    X_train = train[exogenous_vars]
    X_test = test[exogenous_vars]

    # Ajustar modelo ARIMAX com auto_arima
    arimax_model = auto_arima(y_train, exogenous=X_train, seasonal=False, trace=False,
                               error_action='ignore', suppress_warnings=True)
    arimax_order = arimax_model.order

    # Previsão dinâmica com atualização a cada passo
    dynamic_arimax_forecast = []
    history_y = list(y_train)
    history_X = list(X_train.values)

    for t in range(len(y_test)):
        # Ajusta SARIMAX com histórico até o ponto atual
        model = SARIMAX(history_y, exog=np.array(history_X), order=arimax_order)
        fitted_model = model.fit(disp=False)

        # Previsão de 1 passo à frente
        next_x = X_test.iloc[t].values.reshape(1, -1)
        forecast = fitted_model.forecast(steps=1, exog=next_x)
        dynamic_arimax_forecast.append(forecast[0])

        # Atualiza histórico com valores reais
        history_y.append(y_test.iloc[t])
        history_X.append(X_test.iloc[t].values)

    # Calcular MAE
    mae = mean_absolute_error(y_test, dynamic_arimax_forecast)

    # Retornar apenas o MAE para uso na função fitness
    return mae


# Implementacao usando Arimax na funcao fitness

In [31]:
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_absolute_error
from pyswarm import pso
import numpy as np
import pandas as pd

# 📌 Garantir que X_train_scaled e X_test_scaled são DataFrames
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)


# Função fitness para o PSO usando ARIMAX e retorno do MAE
def arimax_feature_selection_fitness(mask, df, tecnica_nome='ARIMAX PSO'):
    mask = np.round(mask).astype(int)

    # Penalizar se mais de 5 ou nenhuma feature selecionada
    num_selected = mask.sum()
    if num_selected > 5 or num_selected == 0:
        return 1e6

    # Obter lista de features selecionadas
    exogenous_vars = df.columns[1:][mask == 1].tolist()  # assumindo 'Inflacao_CPI' é a 1ª coluna

    # Rodar modelo ARIMAX e retornar MAE
    mae = executar_modelo_arimax(df, exogenous_vars, tecnica_nome)
    return mae


# Setup do PSO
num_features = df.shape[1] - 1  # número de colunas exceto 'Inflacao_CPI'
lb = [0] * num_features
ub = [1] * num_features

# Executa PSO para selecionar as melhores features para o ARIMAX
best_mask, _ = pso(
    arimax_feature_selection_fitness,
    lb, ub,
    args=(df,),
    swarmsize=10,
    maxiter=20,
    debug=True
)

# Ajustar máscara e garantir no máximo 5 features selecionadas
best_mask = np.round(best_mask).astype(int)
if best_mask.sum() > 5:
    indices = np.argsort(-best_mask)[:5]  # top 5 ativadas
    new_mask = np.zeros_like(best_mask)
    new_mask[indices] = 1
    best_mask = new_mask

selected_features = df.columns[1:][best_mask == 1]
print("🔹 Melhor conjunto de features selecionado pelo PSO (máx. 5):", list(selected_features))


No constraints given.
Best after iteration 1: [0.22893838 0.0776902  0.05269262 0.20694811 0.49249849 0.72249272
 0.72327899 0.51679256 0.07068316 0.65405749 0.73663293 0.00407599
 0.17815402 0.95055015 0.87038264 0.81564435 0.80743177 0.92041979] 1000000.0
Best after iteration 2: [0.22893838 0.0776902  0.05269262 0.20694811 0.49249849 0.72249272
 0.72327899 0.51679256 0.07068316 0.65405749 0.73663293 0.00407599
 0.17815402 0.95055015 0.87038264 0.81564435 0.80743177 0.92041979] 1000000.0
Best after iteration 3: [0.22893838 0.0776902  0.05269262 0.20694811 0.49249849 0.72249272
 0.72327899 0.51679256 0.07068316 0.65405749 0.73663293 0.00407599
 0.17815402 0.95055015 0.87038264 0.81564435 0.80743177 0.92041979] 1000000.0
Best after iteration 4: [0.22893838 0.0776902  0.05269262 0.20694811 0.49249849 0.72249272
 0.72327899 0.51679256 0.07068316 0.65405749 0.73663293 0.00407599
 0.17815402 0.95055015 0.87038264 0.81564435 0.80743177 0.92041979] 1000000.0
Best after iteration 5: [0.2289383